# Keys

> Type into the focused app (macOS). Used by `iverson-receive`.


In [ ]:
#| default_exp keys


In [ ]:
#| export
import subprocess, sys


In [ ]:
#| export
import threading, time

try:
    import Quartz as Q
    from ApplicationServices import AXIsProcessTrusted, AXIsProcessTrustedWithOptions
    available = sys.platform == 'darwin'
except Exception:
    available = False

CODES = dict(space=49, **{'return':36}, tab=48, backspace=51, escape=53, left=123, right=124, down=125, up=126)

# Hold-to-talk keys (macOS keycodes). Left/right variants listed where needed.
TALK_KEY_CODES = {
    'ctrl': (59, 62), 'control': (59, 62),
    'option': (58, 61), 'alt': (58, 61),
    'command': (55, 54), 'cmd': (55, 54),
    'shift': (56, 60),
    'space': (49,),
    'caps': (57,), 'capslock': (57,),
    'fn': (63,),
}
TALK_KEY_LABELS = [
    ('ctrl', 'Control'),
    ('option', 'Option'),
    ('command', 'Command'),
    ('shift', 'Shift'),
    ('space', 'Space'),
    ('caps', 'Caps Lock'),
    ('fn', 'Fn'),
]

def trusted(): return available and bool(AXIsProcessTrusted())
def status(): return dict(available=available, type=trusted())

def prompt():
    "Show macOS Accessibility permission prompt and open Settings if needed."
    if not available: return
    if not trusted(): AXIsProcessTrustedWithOptions({'AXTrustedCheckOptionPrompt': True})
    if not trusted():
        subprocess.run(['open', 'x-apple.systempreferences:com.apple.preference.security?Privacy_Accessibility'])

def open_bluetooth():
    "Open macOS Bluetooth settings so receive can connect to iverson.io."
    if sys.platform != 'darwin': return
    subprocess.run(['open', 'x-apple.systempreferences:com.apple.BluetoothSettings'], check=False)

def _post(code, text=None):
    src = Q.CGEventSourceCreate(Q.kCGEventSourceStateHIDSystemState)
    for down in (True, False):
        e = Q.CGEventCreateKeyboardEvent(src, code, down)
        Q.CGEventSetFlags(e, 0)
        if text: Q.CGEventKeyboardSetUnicodeString(e, len(text.encode('utf-16-le')) // 2, text)
        Q.CGEventPost(Q.kCGHIDEventTap, e)

def type_key(k):
    "Type a named special key, or the characters of `k`, into the focused app. False if macOS won't allow it."
    if not trusted(): return False
    if k in CODES: _post(CODES[k])
    else: _post(0, k)
    return True

def normalize_talk_key(name):
    "Canonical talk-key id (`ctrl`, `option`, …)."
    aliases = {'control':'ctrl','ctrl':'ctrl','alt':'option','option':'option','cmd':'command','command':'command',
               'capslock':'caps','caps':'caps','shift':'shift','space':'space','fn':'fn'}
    return aliases.get((name or 'ctrl').strip().lower(), 'ctrl')

class TalkGate:
    """Hold talk key → speak one code → release.

    Types each accepted code immediately while armed. Short post-release grace
    catches STT lag for the current press only. Dedupes bursty duplicate notifies.
    """
    def __init__(self, key='ctrl', grace=3.0, dedupe=1.2):
        self.key = normalize_talk_key(key)
        self.grace = grace
        self.dedupe = dedupe
        self.held = False
        self._armed_until = 0.0
        self._lock = threading.Lock()
        self._running = False
        self._thread = None
        self._on_flush = None
        self._last_typed = None
        self._last_typed_at = 0.0

    @property
    def armed(self):
        with self._lock:
            return self.held or time.monotonic() <= self._armed_until

    def set_flush(self, fn):
        "`fn(list[str])` called with payloads to type (usually one code)."
        self._on_flush = fn

    def accept(self, text):
        "If armed, type `text` immediately (skip near-duplicate repeats). Returns True if typed."
        now = time.monotonic()
        with self._lock:
            live = self.held or now <= self._armed_until
            if not live:
                return False
            # Drop STT duplicates that arrive in a burst (same code twice/thrice).
            if text == self._last_typed and (now - self._last_typed_at) < self.dedupe:
                print(f'[iverson-talk] skip duplicate {text!r}', flush=True)
                return True
            self._last_typed = text
            self._last_typed_at = now
            # After release: one code closes the window so late STT can't dump a burst.
            if not self.held:
                self._armed_until = 0.0
        if self._on_flush:
            self._on_flush([text])
        return True

    def _set_held(self, down):
        with self._lock:
            was = self.held
            self.held = down
            if down:
                # New press: fresh window; clear dedupe so the same code can be typed again.
                self._armed_until = 0.0
                self._last_typed = None
                self._last_typed_at = 0.0
            elif was:
                self._armed_until = time.monotonic() + self.grace
        if down and not was:
            print(f'[iverson-talk] {self.key} DOWN — armed', flush=True)
        if not down and was:
            print(f'[iverson-talk] {self.key} UP — armed {self.grace:.0f}s more', flush=True)

    def _key_down(self):
        "True if any mapped talk keycode is currently pressed (HID state)."
        codes = TALK_KEY_CODES.get(self.key, TALK_KEY_CODES['ctrl'])
        return any(Q.CGEventSourceKeyState(Q.kCGEventSourceStateHIDSystemState, c) for c in codes)

    def start(self):
        "Poll HID key state for the talk key. More reliable than an event tap (no Input Monitoring)."
        if not available or self._thread is not None: return False
        self._running = True

        def poll():
            while self._running:
                try:
                    down = self._key_down()
                    if down != self.held:
                        self._set_held(down)
                except Exception:
                    pass
                time.sleep(0.03)

        self._thread = threading.Thread(target=poll, daemon=True, name='iverson-talk-key')
        self._thread.start()
        return True

    def stop(self):
        self._running = False
        self._thread = None


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
